In [7]:
import pandas as pd

train = pd.read_csv("C:/Users/kirub/Downloads/train.csv/train.csv",low_memory=False)
store = pd.read_csv("C:/Users/kirub/Downloads/store.csv")

df = train.merge(store, on="Store")

print(df.head())
print(df.columns)

   Store  DayOfWeek        Date  Sales  Customers  Open  Promo StateHoliday  \
0      1          5  2015-07-31   5263        555     1      1            0   
1      2          5  2015-07-31   6064        625     1      1            0   
2      3          5  2015-07-31   8314        821     1      1            0   
3      4          5  2015-07-31  13995       1498     1      1            0   
4      5          5  2015-07-31   4822        559     1      1            0   

   SchoolHoliday StoreType Assortment  CompetitionDistance  \
0              1         c          a               1270.0   
1              1         a          a                570.0   
2              1         a          a              14130.0   
3              1         c          c                620.0   
4              1         a          a              29910.0   

   CompetitionOpenSinceMonth  CompetitionOpenSinceYear  Promo2  \
0                        9.0                    2008.0       0   
1                   

In [9]:

df = df[df['Open'] == 1]   # remove closed stores
df = df.dropna()           # remove missing values

In [10]:
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['DayOfWeek'] = df['Date'].dt.dayofweek

In [11]:
df['StoreType'] = df['StoreType'].astype('category').cat.codes
df['Assortment'] = df['Assortment'].astype('category').cat.codes
df['StateHoliday'] = df['StateHoliday'].astype('category').cat.codes

In [12]:
df = df.sort_values(['Store', 'Date'])

df['Sales_Lag1'] = df.groupby('Store')['Sales'].shift(1)

df = df.dropna()

In [13]:
print(df.head())
print(df.shape)

         Store  DayOfWeek       Date  Sales  Customers  Open  Promo  \
1013866      2          3 2013-01-03   4159        555     1      0   
1012751      2          4 2013-01-04   4484        574     1      0   
1011636      2          5 2013-01-05   2342        324     1      0   
1009406      2          0 2013-01-07   6775        763     1      1   
1008291      2          1 2013-01-08   6318        685     1      1   

         StateHoliday  SchoolHoliday  StoreType  ...  \
1013866             0              1          0  ...   
1012751             0              1          0  ...   
1011636             0              0          0  ...   
1009406             0              0          0  ...   
1008291             0              0          0  ...   

         CompetitionOpenSinceMonth  CompetitionOpenSinceYear  Promo2  \
1013866                       11.0                    2007.0       1   
1012751                       11.0                    2007.0       1   
1011636             

In [14]:
df = df.drop(columns=['PromoInterval'])

In [15]:
features = [
    'Month', 'Day', 'DayOfWeek',
    'Promo', 'SchoolHoliday',
    'StoreType', 'Assortment',
    'CompetitionDistance',
    'Sales_Lag1'
]

X = df[features]
y = df['Sales']

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

In [20]:
from sklearn.metrics import mean_squared_error

baseline_preds = X_test['Sales_Lag1']

baseline_rmse = mean_squared_error(y_test, baseline_preds, squared=False)

print("Baseline RMSE:", baseline_rmse)


Baseline RMSE: 2061.2865975321884


C:\Users\kirub\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [18]:
import xgboost as xgb

model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

rmse = mean_squared_error(y_test, preds, squared=False)

print("Model RMSE:", rmse)

Model RMSE: 1324.315995420551


C:\Users\kirub\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
